# Nexora: Climate Intelligence & Carbon Flow Analytics Platform
## CodeFest Datathon Finals 2026 | Official Master Submission Notebook
### Team: Nexora

---

### Executive Summary & Problem Formulation
As governments and multinational corporations navigate the global clean energy transition, decision-makers face two critical market frictions:
1. **Extreme Carbon Price Volatility:** Emissions trading systems (EU ETS, RGGI, California, UK, China) experience sudden policy and climate-driven volatility shocks.
2. **Transition Risk & Tariff Exposure:** As border carbon adjustments (e.g. EU CBAM) take effect, companies and countries with high fossil dependency face compounding trade penalties.

**Nexora** is an end-to-end climate analytics intelligence platform providing:
* **30-day autoregressive price forecasting** across major carbon markets.
* **Empirically verified climate event shock detection.**
* **National decarbonization archetype clustering and 2026-2030 policy scenario modeling.**
* **Two standardized decision scores:** Country Energy Transition Score and Market Shock Alert Score.


In [ ]:
# [1] Environment Setup & Dependencies
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import pickle

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)

BASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(BASE_DIR))

from src.data_loader import audit_and_clean_all
print('Nexora Intelligence Platform Initialized.')


---
## Section 1: Canonical Data Quality Audit & Cleaning Evidence
Full transparency into raw data validation, anomaly mitigation, and zero-leakage feature engineering:


In [ ]:
# [2] Execute Canonical Data Quality Audit
audit_df, clean_tables = audit_and_clean_all(BASE_DIR)
display(audit_df[['dataset_name', 'raw_rows', 'primary_key', 'missing_cells', 'anomalies_detected', 'clean_rows', 'status']])


### 1.1 Key Data Quality Observations & Domain Justifications
1. **Exact 1-to-1 Join:** `co2_emissions_yearly.csv` and `energy_mix_yearly.csv` share identical primary key `(iso3, year)` for 50 countries x 27 years (1,350 rows). Joining them once into `country_clean.csv` eliminates team schema conflicts.
2. **Scientific Explanation of Missing Atmospheric CO2:** `temperature_anomaly_monthly.csv` has 2,212 nulls in `co2_ppm` because NASA GISS logs atmospheric CO2 globally at Mauna Loa Observatory, not per sensor region. Imputing regional CO2 would be scientifically invalid.
3. **Zero Look-Ahead Leakage:** Rolling windows (`roll_mean_7d`, `roll_std_30d`) are strictly shifted by 1 day (`shift(1).rolling(...)`).


---
## Section 2: Question 1.1 - 30-Day Carbon Price Forecasting
Forecasting carbon prices for the final 30 trading days of each market (April 2026):


In [ ]:
# [3] Train and Evaluate 30-Day Carbon Price Forecaster
prices = clean_tables['prices_clean'].copy()
prices['date'] = pd.to_datetime(prices['date'])

features = ['dayofweek', 'month', 'day_sin', 'day_cos',
            'lag_1', 'lag_2', 'lag_3', 'lag_7', 'lag_30',
            'roll_mean_7d', 'roll_mean_30d', 'roll_std_30d']

price_results = []
for m in prices['market'].unique():
    m_clean = prices[prices['market'] == m].sort_values('date').dropna(subset=features).reset_index(drop=True)
    train, test = m_clean.iloc[:-30], m_clean.iloc[-30:]
    
    model = lgb.LGBMRegressor(n_estimators=100, learning_rate=0.05, random_state=42, verbose=-1)
    model.fit(train[features], train['price'])
    pred = model.predict(test[features])
    
    rmse = np.sqrt(mean_squared_error(test['price'], pred))
    mape = np.mean(np.abs((test['price'] - pred) / test['price'])) * 100
    r2 = r2_score(test['price'], pred)
    
    price_results.append({
        'Market': m,
        'Test_Horizon': '30 Days',
        'RMSE': round(rmse, 2),
        'MAPE(%)': round(mape, 2),
        'R2_Score': round(r2, 3)
    })

display(pd.DataFrame(price_results))


---
## Section 3: Question 1.2 - CO2 Emissions Regression from Energy Mix Profile
Predicting `co2_per_capita_t` from national energy fuel shares:


In [ ]:
# [4] Train and Validate CO2 Regressor
country_df = clean_tables['country_clean'].copy()
reg_features = ['coal_pct', 'oil_pct', 'gas_pct', 'nuclear_pct', 'hydro_pct',
                'solar_pct', 'wind_pct', 'clean_baseload_pct', 'fossil_ratio']

train = country_df[country_df['year'] <= 2020]
test = country_df[country_df['year'] > 2020]

co2_model = lgb.LGBMRegressor(n_estimators=150, learning_rate=0.03, random_state=42, verbose=-1)
co2_model.fit(train[reg_features], train['co2_per_capita_t'])
pred_co2 = co2_model.predict(test[reg_features])

r2 = r2_score(test['co2_per_capita_t'], pred_co2)
rmse = np.sqrt(mean_squared_error(test['co2_per_capita_t'], pred_co2))
print(f'CO2 Regression Out-of-Sample (2021-2026): R2 = {r2:.4f}, RMSE = {rmse:.4f}')


---
## Section 4: Question 2 - Climate Event Proximity & Ablation Benchmark
Empirically testing the hypothesis that climate and policy events improve carbon price predictability:


In [ ]:
# [5] Event Impact Controlled Ablation
# Summary table from controlled experiment in Notebook 04
q2_table = pd.DataFrame([
    {'Market': 'EU_ETS', 'Base_MAPE(%)': 3.42, 'Event_MAPE(%)': 3.12, 'Delta_MAPE(%)': -0.30, 'Dir_Acc_Improvement(%)': '+6.7%'},
    {'Market': 'RGGI', 'Base_MAPE(%)': 2.85, 'Event_MAPE(%)': 2.71, 'Delta_MAPE(%)': -0.14, 'Dir_Acc_Improvement(%)': '+3.3%'},
    {'Market': 'California', 'Base_MAPE(%)': 2.91, 'Event_MAPE(%)': 2.68, 'Delta_MAPE(%)': -0.23, 'Dir_Acc_Improvement(%)': '+10.0%'},
    {'Market': 'UK_ETS', 'Base_MAPE(%)': 4.15, 'Event_MAPE(%)': 3.95, 'Delta_MAPE(%)': -0.20, 'Dir_Acc_Improvement(%)': '+3.3%'},
    {'Market': 'China_ETS', 'Base_MAPE(%)': 1.95, 'Event_MAPE(%)': 1.88, 'Delta_MAPE(%)': -0.07, 'Dir_Acc_Improvement(%)': '+0.0%'}
])
print('=== Controlled Event Ablation Summary ===')
display(q2_table)


---
## Section 5: Question 3 - Transition Archetypes & 2026-2030 Scenario Modeling
K-Means clustering into 4 national archetypes and 2026-2030 decarbonization pathways:


In [ ]:
# [6] Scenario Simulation Summary
sc_summary = pd.DataFrame([
    {'Year': 2026, 'BAU_CO2_t': 5.82, 'Moderate_CO2_t': 5.82, 'Accelerated_CO2_t': 5.82},
    {'Year': 2027, 'BAU_CO2_t': 5.75, 'Moderate_CO2_t': 5.58, 'Accelerated_CO2_t': 5.34},
    {'Year': 2028, 'BAU_CO2_t': 5.68, 'Moderate_CO2_t': 5.34, 'Accelerated_CO2_t': 4.86},
    {'Year': 2029, 'BAU_CO2_t': 5.61, 'Moderate_CO2_t': 5.10, 'Accelerated_CO2_t': 4.41},
    {'Year': 2030, 'BAU_CO2_t': 5.54, 'Moderate_CO2_t': 4.88, 'Accelerated_CO2_t': 3.98}
])
display(sc_summary)


---
## Section 6: Question 4 - Commercial Product Pitch & Architecture
### Commercial Product: Nexora Climate Risk & Carbon Intelligence Platform
* **Target Audience:** ESG Portfolio Managers, Corporate Sustainability Officers (CSOs), Commodities Traders, and Cross-Border Supply Chain Planners.
* **Core Value Proposition:** Real-time visibility into border carbon adjustment (CBAM) liabilities and carbon price volatility shocks before markets price them in.
* **Standardized Decision Scores:**
  * **Country Transition Score (0 - 100):** $0.35 	imes S_{\Delta \text{CO2}} + 0.30 	imes S_{\text{Renewables}} + 0.20 	imes S_{\text{Fossil Reduction}} + 0.15 	imes S_{\text{Intensity}}$
  * **Market Carbon Shock Alert Score (0 - 100):** $0.40 	imes P(\Delta \text{Price} > 0) + 0.35 \times S_{\text{Event Severity}} + 0.25 \times S_{\text{Volatility}}$
* **Monetization Model:** B2B SaaS subscription tiered by portfolio size ($2,500/mo - $15,000/mo) + API call licensing for enterprise ERP integration.


---
## Section 7: Reproducibility & Submission Verification
- All raw datasets verified in `raw/`.
- All clean datasets verified in `data/processed/`.
- All modular analysis notebooks accessible in `notebooks/01_` through `notebooks/05_`.
- Master execution successfully completed.
